In [8]:
import cv2
import numpy as np
import os

# Function to add salt & pepper noise
def add_salt_pepper_noise(image, prob):
    noisy = image.copy()
    total_pixels = int(prob * image.shape[0] * image.shape[1])
    for _ in range(total_pixels):
        row = np.random.randint(0, image.shape[0])
        col = np.random.randint(0, image.shape[1])
        color = np.random.choice([0, 255])
        noisy[row, col] = [color, color, color]
    return noisy

# Function to show and save image
def show_and_save(title, image, folder):
    cv2.imshow(title, image)
    cv2.waitKey(0)
    filename = os.path.join(folder, f"{title.replace(' ', '_').lower()}.jpg")
    cv2.imwrite(filename, image)

# Create output folder
output_folder = 'output_images'
os.makedirs(output_folder, exist_ok=True)

# Load the image
img = cv2.imread('images/NailPolish.jpg')

# Check if image loaded successfully
if img is None:
    raise FileNotFoundError("Image not found. Please check the path and try again.")

# Show and save original image
show_and_save('Original Image', img, output_folder)

# Create noisy images
x = float(input("noise level? "))
noisy_x = add_salt_pepper_noise(img, x)
noisy_10 = add_salt_pepper_noise(img, 0.10)
noisy_25 = add_salt_pepper_noise(img, 0.25)
show_and_save(f'Salt & Pepper Noise p={x}', noisy_x, output_folder)
show_and_save('Salt & Pepper Noise p=0.10', noisy_10, output_folder)
show_and_save('Salt & Pepper Noise p=0.25', noisy_25, output_folder)

# Define kernel sizes
kernel_sizes = [(3, '3x3'), (25, '25x25')]

# Apply filters for both noise levels and kernel sizes
for noisy_img, prob in [(noisy_10, 'p=0.10'), (noisy_25, 'p=0.25')]:
    for ksize, label in kernel_sizes:
        # Box Filter
        box = cv2.blur(noisy_img, (ksize, ksize))
        show_and_save(f'Box Filter {label} {prob}', box, output_folder)

        # Median Filter
        median = cv2.medianBlur(noisy_img, ksize)
        show_and_save(f'Median Filter {label} {prob}', median, output_folder)

        # Bilinear Filter (Gaussian Blur)
        bilinear = cv2.GaussianBlur(noisy_img, (ksize, ksize), 0)
        show_and_save(f'Bilinear Filter {label} {prob}', bilinear, output_folder)

        # Bilateral Filter
        bilateral = cv2.bilateralFilter(noisy_img, ksize, 75, 75)
        show_and_save(f'Bilateral Filter {label} {prob}', bilateral, output_folder)

# Close all windows
cv2.destroyAllWindows()

noise level?  0.5


In [7]:
import cv2
import numpy as np

# Load the image
img = cv2.imread('images/NailPolish.jpg')
img = add_salt_pepper_noise(img, 0.1)

# Choose a pixel (e.g., row=100, col=100)
row, col = 100, 100

# Extract 3x3 neighborhood
neighborhood = img[row-1:row+2, col-1:col+2]

# Separate B, G, R channels
B_vals = neighborhood[:, :, 0].flatten()
G_vals = neighborhood[:, :, 1].flatten()
R_vals = neighborhood[:, :, 2].flatten()

print("Original Pixel and Neighborhood Values:")
print(f"B channel: {B_vals}")
print(f"G channel: {G_vals}")
print(f"R channel: {R_vals}")

# Calculate expected de-noised values
B_box = int(np.mean(B_vals))         # Box filter (average)
G_median = int(np.median(G_vals))    # Median filter
R_bilinear = int(cv2.GaussianBlur(neighborhood[:, :, 2], (3, 3), 0)[1, 1])  # Bilinear (Gaussian)

print("\nExpected De-noised Pixel Values:")
print(f"B (Box Filter): {B_box}")
print(f"G (Median Filter): {G_median}")
print(f"R (Bilinear Filter): {R_bilinear}")

# Apply actual filters to the image
B_filtered = cv2.blur(img[:, :, 0], (3, 3))
G_filtered = cv2.medianBlur(img[:, :, 1], 3)
R_filtered = cv2.GaussianBlur(img[:, :, 2], (3, 3), 0)

actual_B = B_filtered[row, col]
actual_G = G_filtered[row, col]
actual_R = R_filtered[row, col]

print("\nActual Filtered Pixel Values:")
print(f"B (Box Filter): {actual_B}")
print(f"G (Median Filter): {actual_G}")
print(f"R (Bilinear Filter): {actual_R}")

Original Pixel and Neighborhood Values:
B channel: [138 150 167 255 148 155 138 147 138]
G channel: [164 176 193 255 174 181 164 173 164]
R channel: [194 206 223 255 204 211 194 203 194]

Expected De-noised Pixel Values:
B (Box Filter): 159
G (Median Filter): 174
R (Bilinear Filter): 211

Actual Filtered Pixel Values:
B (Box Filter): 160
G (Median Filter): 174
R (Bilinear Filter): 211
